# Notebook 10 — Post-fit Outputs and Visualisation

After optimisation, `phoscrosstalk.analysis` writes a standardised set of output
files and produces diagnostic plots. This notebook explains each output, shows its
column format, and demonstrates how to visualise fitted vs observed trajectories.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLE_DIR = PROJECT_ROOT / "notebooks" / "sample_data"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1  Output file inventory

| File | Format | Content |
|---|---|---|
| `pareto_front_with_J.tsv` | TSV | Per-start loss components `[f1, f2, f3, f4, J]` |
| `pareto_points.tsv` | TSV | Per-start theta vectors |
| `pareto_front.npz` | NumPy | Same data in binary form |
| `pareto_stats.tsv` | TSV | Summary statistics across starts |
| `fit_timeseries.tsv` | TSV | Long-format fitted trajectories at observed time points |
| `fit_timeseries_dense.tsv` | TSV | Dense-grid (200 points) trajectories |
| `protein_fit_timeseries.tsv` | TSV | Protein abundance trajectories |
| `theta_best.npy` | NumPy | Best theta vector |
| `time_axes.json` | JSON | Time arrays used during fitting |

In [ ]:
from phoscrosstalk.config import ModelDims
from phoscrosstalk.data_loader import (
    load_site_data, load_rna_data, load_kinase_site_matrix,
    load_tf_network, build_tf_prot_weights, apply_scaling, row_normalize,
)

TIMEPOINTS = list(range(1, 15))   # 14 time points x1..x14

# ── phosphosite + protein abundance ─────────────────────────────────────────
sites, proteins, site_prot_idx, positions, t_phos, Y, A_data, A_proteins = \
    load_site_data(str(SAMPLE_DIR / "protephospho.csv"), TIMEPOINTS)

K = len(proteins)
N = len(sites)
T = len(t_phos)
print(f"proteins : {proteins}  (K={K})")
print(f"sites    : {sites}  (N={N})")
print(f"t_phos   : {t_phos}  (T={T})")
print(f"Y        : {Y.shape}   (N × T  phosphosite data)")
print(f"A_data   : {A_data.shape}  (K × T  protein abundance)")

# ── mRNA ─────────────────────────────────────────────────────────────────────
gene_ids, t_rna, rna_matrix = load_rna_data(
    str(SAMPLE_DIR / "mrna.csv"), timepoints=TIMEPOINTS
)
print(f"gene_ids : {gene_ids}  (n_genes={len(gene_ids)})")
print(f"rna_matrix: {rna_matrix.shape}  (n_genes × T)")

# ── kinase-site matrix ───────────────────────────────────────────────────────
K_site_kin, kinases = load_kinase_site_matrix(
    str(SAMPLE_DIR / "kinase_sites.tsv"), sites
)
M = len(kinases)
print(f"kinases  : {kinases}  (M={M})")
print(f"K_site_kin: {K_site_kin.shape}  (N × M)")

# ── kinase → protein index ───────────────────────────────────────────────────
kin_to_prot_idx = np.array([proteins.index(k) for k in kinases], dtype=int)
print(f"kin_to_prot_idx: {kin_to_prot_idx}")

# ── TF network ───────────────────────────────────────────────────────────────
tf_net = load_tf_network(str(SAMPLE_DIR / "tf_mrna.csv"), gene_ids=gene_ids)
tf_prot_weights = build_tf_prot_weights(tf_net, gene_ids, proteins)
print(f"tf_prot_weights: {tf_prot_weights.shape}  (K × n_genes)")

# ── scaled data ──────────────────────────────────────────────────────────────
P_scaled, _, _  = apply_scaling(Y)
A_scaled, _, _  = apply_scaling(A_data)
dims = ModelDims(K=K, M=M, N=N)

In [ ]:
from phoscrosstalk.optimization import create_bounds, build_parameter_labels
from phoscrosstalk.weighting import build_weight_matrices
from phoscrosstalk.derived_rates import make_k_act_fn, make_s_prod_fn
from phoscrosstalk.simulation import simulate
from phoscrosstalk.mechanisms import decode_theta

Cg = np.zeros((N, N)); Cl = np.zeros((N, N))
R_kin = row_normalize(K_site_kin.T); L_alpha = np.zeros((M, M))
receptor_mask_prot = np.zeros(K); receptor_mask_kin = np.zeros(M)

k_act_fn = make_k_act_fn(t_rna=t_rna, rna_data=rna_matrix,
                         tf_prot_weights=tf_prot_weights, K=K)
s_prod_fn = make_s_prod_fn(
    t_protein=t_phos, Y_data=P_scaled,
    R_kin_site=row_normalize(K_site_kin.T),
    kin_to_prot_idx=kin_to_prot_idx, K=K, M=M,
)
xl, xu, dim = create_bounds(K, M, N)
print(f"K={K}, M={M}, N={N}, dim={dim}")


## 2  Create synthetic best theta and run simulation

In [ ]:
rng = np.random.default_rng(7)
theta_best = rng.uniform(xl, xu)

P_sim, A_sim, S_sim, Kdyn_sim = simulate(
    t_arr=t_phos, P_data0=P_scaled, A_data0=A_scaled,
    theta=theta_best, Cg=Cg, Cl=Cl,
    site_prot_idx=site_prot_idx, K_site_kin=K_site_kin, R=R_kin,
    L_alpha=L_alpha, kin_to_prot_idx=kin_to_prot_idx,
    receptor_mask_prot=receptor_mask_prot, receptor_mask_kin=receptor_mask_kin,
    mechanism="dist", full_output=True,
    k_act_fn=k_act_fn, s_prod_fn=s_prod_fn,
)
print("P_sim:", P_sim.shape, "  A_sim:", A_sim.shape)
print("S_sim:", S_sim.shape, "  Kdyn_sim:", Kdyn_sim.shape)

np.save(OUTPUT_DIR / "theta_best.npy", theta_best)
print("Saved theta_best.npy")


## 3  `fit_timeseries.tsv` — long-format structure

| Column | Description |
|---|---|
| `entity_type` | `'phosphosite'` or `'protein'` |
| `entity` | Site name (e.g. `EGFR_Y1068`) or protein name |
| `site` | Phosphosite identifier |
| `protein` | Corresponding protein name |
| `time` | Time point value |
| `value` | Scaled intensity / abundance |
| `series_type` | `'observed'` or `'simulated'` |
| `source` | Data origin tag |
| `interpolation_method` | `'piecewise_constant'` or `'linear'` |

In [ ]:
rows = []
for i, site in enumerate(sites):
    pname = proteins[site_prot_idx[i]]
    for t_idx, t_val in enumerate(t_phos):
        for series, values in [("observed", P_scaled), ("simulated", P_sim)]:
            rows.append(dict(
                entity_type="phosphosite", entity=site, site=site,
                protein=pname, time=float(t_val),
                value=float(values[i, t_idx]),
                series_type=series, source="sample_data",
                interpolation_method="piecewise_constant",
            ))
for p_idx, pname in enumerate(proteins):
    for t_idx, t_val in enumerate(t_phos):
        for series, values in [("observed", A_scaled), ("simulated", A_sim)]:
            rows.append(dict(
                entity_type="protein", entity=pname, site="",
                protein=pname, time=float(t_val),
                value=float(values[p_idx, t_idx]),
                series_type=series, source="sample_data",
                interpolation_method="piecewise_constant",
            ))

fit_ts = pd.DataFrame(rows)
fit_ts.to_csv(OUTPUT_DIR / "fit_timeseries.tsv", sep="\t", index=False)
print("fit_timeseries.tsv  shape:", fit_ts.shape)
print(fit_ts.head(4).to_string())


## 4  Plot fitted vs observed phosphosites

In [ ]:
phos_df = fit_ts[fit_ts["entity_type"] == "phosphosite"]

fig, axes = plt.subplots(3, 3, figsize=(13, 8), sharex=True)
axes = axes.ravel()
for i, site in enumerate(sites):
    ax = axes[i]
    obs = phos_df[(phos_df["entity"] == site) & (phos_df["series_type"] == "observed")]
    sim = phos_df[(phos_df["entity"] == site) & (phos_df["series_type"] == "simulated")]
    ax.plot(obs["time"], obs["value"], "o--", ms=4, color="tab:blue",   label="obs")
    ax.plot(sim["time"], sim["value"], "-",   lw=2, color="tab:orange", label="sim")
    ax.set_title(site, fontsize=9)
    if i == 0:
        ax.legend(fontsize=7)
fig.suptitle("Fitted vs Observed — Phosphosites  (synthetic theta)", y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "10_phospho_fit.png", dpi=100)
plt.show()
print("Saved 10_phospho_fit.png")


## 5  Plot fitted vs observed proteins

In [ ]:
prot_df = fit_ts[fit_ts["entity_type"] == "protein"]

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for p_idx, pname in enumerate(proteins):
    ax = axes[p_idx]
    obs = prot_df[(prot_df["entity"] == pname) & (prot_df["series_type"] == "observed")]
    sim = prot_df[(prot_df["entity"] == pname) & (prot_df["series_type"] == "simulated")]
    ax.plot(obs["time"], obs["value"], "o--", ms=4, color="tab:blue",   label="obs")
    ax.plot(sim["time"], sim["value"], "-",   lw=2, color="tab:orange", label="sim")
    ax.set_title(pname); ax.set_xlabel("Time index")
    if p_idx == 0:
        ax.set_ylabel("Abundance (scaled)"); ax.legend()
fig.suptitle("Fitted vs Observed — Protein Abundance  (synthetic theta)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "10_protein_fit.png", dpi=100)
plt.show()
print("Saved 10_protein_fit.png")


## 6  Decode `theta_best` into biological parameters

`decode_theta(theta, K, M, N)` unpacks the log-space theta vector into named
parameter blocks that correspond directly to biochemical rate constants.


In [ ]:
# decode_theta returns natural-scale values (exp already applied internally)
(
    k_deact, d_deg, beta_g, beta_l,
    alpha, kK_act, kK_deact,
    k_off,
    gamma_S_p, gamma_A_S, gamma_A_p, gamma_K_net,
) = decode_theta(theta_best, K, M, N)

param_rows = []
for i, p in enumerate(proteins):
    param_rows += [
        {"entity": p, "param": "k_deact", "value": float(k_deact[i])},
        {"entity": p, "param": "d_deg",   "value": float(d_deg[i])},
    ]
for m, kin in enumerate(kinases):
    param_rows += [
        {"entity": kin, "param": "alpha",    "value": float(alpha[m])},
        {"entity": kin, "param": "kK_act",   "value": float(kK_act[m])},
        {"entity": kin, "param": "kK_deact", "value": float(kK_deact[m])},
    ]
for i, site in enumerate(sites):
    param_rows.append({"entity": site, "param": "k_off", "value": float(k_off[i])})

param_df = pd.DataFrame(param_rows)
print(param_df.pivot_table(index="entity", columns="param", values="value").round(4).to_string())


## 7  Parameter heatmaps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# k_deact, kK_act, etc. are already in natural scale from decode_theta
kin_params = np.column_stack([np.array(alpha), np.array(kK_act), np.array(kK_deact)])
im0 = axes[0].imshow(kin_params, aspect="auto", cmap="YlOrRd")
axes[0].set_title("Kinase parameters (natural scale)")
axes[0].set_yticks(range(M)); axes[0].set_yticklabels(kinases)
axes[0].set_xticks([0, 1, 2]); axes[0].set_xticklabels(["α", "kK_act", "kK_deact"])
plt.colorbar(im0, ax=axes[0])

prot_params = np.column_stack([np.array(k_deact), np.array(d_deg)])
im1 = axes[1].imshow(prot_params, aspect="auto", cmap="Blues")
axes[1].set_title("Protein parameters")
axes[1].set_yticks(range(K)); axes[1].set_yticklabels(proteins)
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(["k_deact", "d_deg"])
plt.colorbar(im1, ax=axes[1])

site_params = np.array(k_off).reshape(-1, 1)
im2 = axes[2].imshow(site_params, aspect="auto", cmap="Greens")
axes[2].set_title("Phosphosite k_off")
axes[2].set_yticks(range(N)); axes[2].set_yticklabels(sites, fontsize=8)
axes[2].set_xticks([0]); axes[2].set_xticklabels(["k_off"])
plt.colorbar(im2, ax=axes[2])

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "10_parameter_heatmaps.png", dpi=100)
plt.show()
print("Saved 10_parameter_heatmaps.png")


## 8  Multi-start loss comparison

`pareto_front_with_J.tsv` is produced by `save_run_results()` in `analysis.py`.
We replicate its structure here with synthetic data.

In [ ]:
n_demo = 5
rng_demo = np.random.default_rng(10)
F_demo = np.abs(rng_demo.normal(loc=0.5, scale=0.15, size=(n_demo, 4)))
J_demo = F_demo.sum(axis=1)
best_demo = int(np.argmin(J_demo))

pareto_demo = pd.DataFrame(F_demo, columns=["f1_phospho","f2_protein","f3_reg","f4_mrna"])
pareto_demo["J_total"] = J_demo
print(pareto_demo.round(4).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
bottoms = np.zeros(n_demo)
for c_idx, (col, lbl) in enumerate(zip(
        ["tab:blue","tab:orange","tab:green","tab:red"],
        ["f1 phospho","f2 protein","f3 reg","f4 mRNA"])):
    ax.bar(range(n_demo), pareto_demo.iloc[:, c_idx],
           bottom=bottoms, color=col, label=lbl, alpha=0.85)
    bottoms += pareto_demo.iloc[:, c_idx].values
ax.axvline(best_demo, color="red", ls="--", lw=2, label=f"best (start {best_demo})")
ax.set_xlabel("Start index"); ax.set_ylabel("Stacked loss J")
ax.set_title("Multi-start loss components (synthetic)"); ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "10_multistart_losses.png", dpi=100)
plt.show()
print("Saved 10_multistart_losses.png")


## 9  `time_axes.json` and dense simulation

`time_axes.json` stores the time arrays used during fitting:

```json
{
  "t_phos": [1, 2, 3, ...],
  "t_prot": [1, 2, 3, ...],
  "t_dense": [1.0, 1.1, 1.2, ...]
}
```

`fit_timeseries_dense.tsv` is produced by simulating on 200 grid points
for smooth visualisation curves.

In [ ]:
import json

t_dense = np.linspace(t_phos[0], t_phos[-1], 200)
time_axes = {
    "t_phos":  t_phos.tolist(),
    "t_prot":  t_phos.tolist(),
    "t_rna":   t_rna.tolist(),
    "t_dense": t_dense.tolist(),
}
with open(OUTPUT_DIR / "time_axes.json", "w") as fh:
    json.dump(time_axes, fh, indent=2)
print("Saved time_axes.json")

P_dense, A_dense = simulate(
    t_arr=t_dense, P_data0=P_scaled, A_data0=A_scaled,
    theta=theta_best, Cg=Cg, Cl=Cl,
    site_prot_idx=site_prot_idx, K_site_kin=K_site_kin, R=R_kin,
    L_alpha=L_alpha, kin_to_prot_idx=kin_to_prot_idx,
    receptor_mask_prot=receptor_mask_prot, receptor_mask_kin=receptor_mask_kin,
    mechanism="dist", k_act_fn=k_act_fn, s_prod_fn=s_prod_fn,
)
print("P_dense:", P_dense.shape, "  (N × 200 time points for smooth plots)")


## 10  Biological plausibility score (`bio_score`)

`bio_score(theta, dims)` evaluates whether the fitted rate constants imply
plausible **half-lives** for each model state:

| State | Expected half-life range |
|---|---|
| mRNA | 15 min – 12 h |
| Protein | 2 h – 96 h |
| Phosphosite | 1 min – 60 min |

A score of 0 = fully plausible; higher = more implausible rate constants.
Stored in `pareto_stats.tsv` alongside loss components.

In [ ]:
from phoscrosstalk.optimization import bio_score

bs = bio_score(theta_best, dims=dims)
print(f"bio_score = {bs:.4f}  (0 = fully plausible, higher = more implausible)")

# Decode for half-life interpretation
# decode_theta already applied exp(); unpack all 12 values
(k_deact2, d_deg2, _, _, _, _, _, k_off2,
 _, _, _, _) = decode_theta(theta_best, K, M, N)

hl_mrna  = np.log(2) / np.array(k_deact2)
hl_prot  = np.log(2) / np.array(d_deg2)
hl_psite = np.log(2) / np.array(k_off2)

print("\nProtein mRNA half-lives (time units):   ", hl_mrna.round(3))
print("Protein abundance half-lives (time units):", hl_prot.round(3))
print("Phosphosite half-lives (time units):      ", hl_psite.round(3))
